In [ ]:
from pathlib import Path
import json
from html import escape
import ipywidgets as widgets
from IPython.display import display, HTML, Image, clear_output

# Temukan root proyek walaupun kernel dimulai dari folder lain di dalam proyek.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/chunks/sus/chunks.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Jalankan notebook dari folder proyek RAG yang memi    Get-ChildItem -Recurse -File |
        Sort-Object Length -Descending |
        Select-Object -First 20 FullName,@{Name="SizeMB";Expression={[math]::Round($_.Length / 1MB, 2)}}liki data/chunks/sus/chunks.json")
CHUNKS_PATH = ROOT / "data/chunks/sus/chunks.json"

# Pembacaan lokal saja. Tidak menjalankan parser, chunking, model, atau jaringan.
def load_chunks():
    document = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
    if not isinstance(document.get("chunks"), list):
        raise ValueError("Field chunks harus berupa list.")
    return document

document = load_chunks()
chunks = document["chunks"]
print(f"Sumber: {CHUNKS_PATH} | {len(chunks)} chunk | metode: {document['chunking']['method']}")


In [ ]:
search = widgets.Text(description="Cari:", placeholder="Heading, teks, atau chunk ID", continuous_update=False,
                      layout=widgets.Layout(width="75%"))
selector = widgets.Dropdown(description="Chunk:", layout=widgets.Layout(width="98%"))
previous = widgets.Button(description="Sebelumnya")
following = widgets.Button(description="Berikutnya")
reload_button = widgets.Button(description="Muat ulang JSON", icon="refresh")
summary = widgets.HTML()
metadata = widgets.HTML()
text_view = widgets.Textarea(disabled=True, layout=widgets.Layout(width="100%", height="420px"))
blocks_view = widgets.Output(layout=widgets.Layout(max_height="650px", overflow="auto"))
json_view = widgets.Textarea(disabled=True, layout=widgets.Layout(width="100%", height="500px"))
overview = widgets.HTML()
tabs = widgets.Tab(children=[text_view, blocks_view, json_view, overview])
for index, title in enumerate(["Teks chunk", "Blok & aset", "JSON chunk", "Ringkasan semua chunk"]):
    tabs.set_title(index, title)


def title(chunk):
    return " > ".join(chunk.get("section_path", [])) or "Pembuka (sebelum heading pertama)"


def table_markup(columns, rows):
    def cells(values, tag):
        return "".join(f"<{tag} style='border:1px solid #888;padding:6px;text-align:left'>{escape(str(v))}</{tag}>" for v in values)
    return ("<div style='overflow-x:auto'><table style='border-collapse:collapse'>"
            + "<thead><tr>" + cells(columns, "th") + "</tr></thead><tbody>"
            + "".join("<tr>" + cells(row, "td") + "</tr>" for row in rows)
            + "</tbody></table></div>")


def render(change=None):
    index = selector.value
    previous.disabled = index is None or list(v for _,v in selector.options).index(index) == 0
    following.disabled = index is None or list(v for _,v in selector.options).index(index) == len(selector.options)-1
    if index is None:
        metadata.value = "<p>Tidak ada chunk yang cocok.</p>"
        text_view.value = json_view.value = ""
        with blocks_view:
            clear_output(wait=False)
        return
    chunk = chunks[index]
    fields = [("ID",chunk["chunk_id"]),("Section",title(chunk)),("Halaman PDF",chunk.get("pages",[])),
              ("Karakter",len(chunk.get("text",""))),("Jenis konten",chunk.get("content_types",[])),
              ("Status",chunk.get("status")),("Akses",chunk.get("access"))]
    metadata.value = "<br>".join(f"<b>{escape(k)}:</b> {escape(str(v))}" for k,v in fields)
    text_view.value = chunk.get("text", "")
    json_view.value = json.dumps(chunk, ensure_ascii=False, indent=2)
    with blocks_view:
        clear_output(wait=True)
        for block in chunk.get("blocks", []):
            display(HTML(f"<hr><b>{escape(block['type'])}</b> ? {escape(block['id'])} ? PDF page {escape(str(block.get('source',{}).get('page')))}"))
            if block["type"] == "table":
                display(HTML(table_markup(block["columns"], block["rows"])))
                for row in block.get("summary_rows", []):
                    print("Ringkasan:", " ".join(v for v in row if v))
                if block.get("continuation_of"):
                    print("Lanjutan dari:", block["continuation_of"])
            elif block["type"] == "image":
                asset = (ROOT / block["asset_path"]).resolve()
                if not asset.is_relative_to(ROOT.resolve()):
                    print("Aset di luar root proyek tidak dibuka:", asset)
                elif asset.is_file():
                    display(Image(filename=str(asset), width=650))
                else:
                    print("Aset tidak ditemukan:", asset)
                print("Kandidat caption:", block.get("caption_text", "Belum tersedia"))
            else:
                display(HTML("<pre style='white-space:pre-wrap'>" + escape(block.get("text", "")) + "</pre>"))


def filter_chunks(change=None):
    query = search.value.strip().casefold()
    options = [(f"{i+1:02d}. {title(c)} ({len(c.get('text','')):,} karakter)",i)
               for i,c in enumerate(chunks)
               if not query or query in (title(c)+" "+c["chunk_id"]+" "+c.get("text","")).casefold()]
    old = selector.value
    selector.options = options
    selector.value = old if old in [v for _,v in options] else (options[0][1] if options else None)
    summary.value = f"<b>{len(options)} dari {len(chunks)} chunk</b> | header-based | tanpa batas ukuran | overlap 0"
    render()


def navigate(step):
    indices = [value for _,value in selector.options]
    if selector.value in indices:
        position = indices.index(selector.value)+step
        if 0 <= position < len(indices):
            selector.value = indices[position]


def reload_data(button=None):
    global document, chunks
    document = load_chunks()
    chunks = document["chunks"]
    overview.value = table_markup(["No", "Section", "Halaman PDF", "Karakter", "Blok", "Status"],
        [[i+1,title(c),str(c.get("pages",[])),len(c.get("text","")),len(c.get("blocks",[])),c.get("status")]
         for i,c in enumerate(chunks)])
    filter_chunks()

selector.observe(render, names="value")
search.observe(filter_chunks, names="value")
previous.on_click(lambda button: navigate(-1))
following.on_click(lambda button: navigate(1))
reload_button.on_click(reload_data)
reload_data()
display(widgets.VBox([search, selector, widgets.HBox([previous,following,reload_button]),
                      summary, metadata, tabs]))
